In [ ]:
import os
import numpy as np
import tifffile

# =====================================================================
# DATASET ARCHIVE CONFIGURATION
# =====================================================================
# Local directory mapping containing multi-channel raw TIFF imaging scans
# Note: Raw scan files are withheld from the repository to protect dataset privacy
DATA_DIR = "./datasets/raw_scans"

TIFF_FILES = [
    "F+B+lambda first 100 um top cornea 10X.tif",
    "F+B+Lambda starting from 500 um cornea 10x.tif",
    "Forward_backward_lambda_scan_lenticule.tif",
    "SMILE 160 um - around the SMILE-  25 mmHg deflated to 23 (5 um interval).tif",
    "SMILE 160 um - 50un right below the plane of the SMILE 25 mmHg deflated to 23 (0.81 um interval).tif",
    "SMILE 160 um - 50un right below the plane of the SMILE 14 mmHg deflated to 14 (0.81 um interval).tif",
    "SMILE 160 um - 50un right below the plane of the SMILE 10 mmHg deflated to 8 (0.81 um interval).tif"
]

# =====================================================================
# CROSS-MODAL CHANNEL EXTRACTION PIPELINE
# =====================================================================
def process_tiff_files(file_list, base_dir=DATA_DIR):
    """
    Parses multi-channel Z-stack TIFF structures, separating raw cross-modal
    microscopy signals into decoupled forward and backward scattering collections.

    Args:
        file_list (list): Collection of target TIFF filename strings to parse.
        base_dir (str): Root file directory containing target files.

    Returns:
        tuple: Co-registered lists of (all_backward_slices, all_forward_slices).
    """
    all_backward_slices = []
    all_forward_slices = []
    total_slice_pairs = 0

    for file_name in file_list:
        file_path = os.path.join(base_dir, file_name)
        print(f"\nProcessing File: {file_name}")

        try:
            img = tifffile.imread(file_path)
        except FileNotFoundError:
            print(f"Skipping: {file_name} not found in directory '{base_dir}'.")
            continue

        print(f"  -> Input Volumetric Shape: {img.shape}")

        img = np.squeeze(img)  # Eliminate singleton artifacts
        if img.ndim != 4:
            raise ValueError(f"Structural Mismatch: Expected tensor dimensions (Z, C, Y, X), got {img.shape}")

        n_stacks, n_channels, Y, X = img.shape
        print(f"  -> Parsed Geometry: {n_stacks} stacks × {n_channels} channels ({Y}×{X})")

        # Sequentially de-interleave multi-channel cross-modal slices per focal position
        for z in range(n_stacks):
            # Channel Mapping Configuration:
            # - Channel Index 1: Backward-scattering structural modality (Input)
            # - Channel Index 4: Forward-scattering ground-truth target (Label)
            all_backward_slices.append(img[z, 1, :, :])
            all_forward_slices.append(img[z, 4, :, :])
            total_slice_pairs += 1

    print(f"\n=====================================================================")
    print(f"CROSS-MODAL EXTRACTION COMPLETE:")
    print(f"=====================================================================")
    print(f"  Total Extracted Slices:   {total_slice_pairs} structural co-registered pairs.")
    print(f"  Backward Array Dimensions: {len(all_backward_slices)} slices.")
    print(f"  Forward Array Dimensions:  {len(all_forward_slices)} slices.")

    return all_backward_slices, all_forward_slices

# =====================================================================
# EXECUTION ROUTINE
# =====================================================================
backward_slices, forward_slices = process_tiff_files(TIFF_FILES)


In [ ]:

import numpy as np
from skimage import exposure
from scipy.ndimage import median_filter

# =====================================================================
# PATCH EXTRACTION METHODOLOGY
# =====================================================================
def extract_patches(image_2d, patch_size, stride):
    """
    Extracts uniform, square patches from a 2D image matrix.

    Args:
        image_2d (np.array): Input 2D NumPy array representing a single slice.
        patch_size (int): Dimension edge length of the square patches to extract.
        stride (int): Step size used to advance the translation extraction window.
                      If stride == patch_size, patches are strictly non-overlapping.

    Returns:
        list: Collection of extracted 2D NumPy array patches.
    """
    H, W = image_2d.shape
    patches = []

    # Iterate over the height and width limits of the target matrix array
    for i in range(0, H - patch_size + 1, stride):
        for j in range(0, W - patch_size + 1, stride):
            patch = image_2d[i:i+patch_size, j:j+patch_size]
            patches.append(patch)

    return patches

# Configure baseline non-overlapping spatial extraction bounds
PATCH_SIZE = 256
STRIDE = 256

# Process arrays into temporary collections
bk_patches_list = []
for img_slice in backward_slices:
    bk_patches_list.extend(extract_patches(img_slice.astype(np.float64), PATCH_SIZE, STRIDE))

fw_patches_list = []
for img_slice in forward_slices:
    fw_patches_list.extend(extract_patches(img_slice.astype(np.float64), PATCH_SIZE, STRIDE))

# Consolidate 2D list matrix arrays into singular 3D tensor configurations
backward_patches = np.stack(bk_patches_list)
forward_patches = np.stack(fw_patches_list)

print(f"Stacked Volumetric Matrices:")
print(f"  -> Backward Channel Array Geometry: {backward_patches.shape}")
print(f"  -> Forward Channel Array Geometry:  {forward_patches.shape}")

# =====================================================================
# GLOBAL DATA PERCENTILE NORMALISATION
# =====================================================================
# Flatten array bounds across all slices to identify absolute outliers
all_pixels_b = np.concatenate([img_slice.flatten() for img_slice in backward_slices])
all_pixels_f = np.concatenate([img_slice.flatten() for img_slice in forward_slices])

# Derive the 1st and 99th intensity boundaries to map clean normalization spreads
p1_b, p99_b = np.percentile(all_pixels_b, (1, 99))
p1_f, p99_f = np.percentile(all_pixels_f, (1, 99))

def normalize_percentile(x, p1, p99):
    """
    Normalizes pixel intensity variations utilizing 1st and 99th percentile distributions.
    Caps high-frequency scanner anomalies and maps structural arrays down to.
    """
    x = (x - p1) / (p99 - p1)
    x = np.clip(x, 0, 1)
    return x.astype(np.float32)

# Deploy normalisation loops across the array targets
backward_norm = [normalize_percentile(p, p1_b, p99_b) for p in backward_patches]
forward_norm  = [normalize_percentile(p, p1_f, p99_f) for p in forward_patches]

# =====================================================================
# GROUND TRUTH CONDITIONING & BINARISATION
# =====================================================================
processed_forward_norm = []

for img_array in forward_norm:
    modified_img_array = img_array.copy()

    # Apply Contrast-Limited Adaptive Histogram Equalisation (CLAHE)
    modified_img_array = exposure.equalize_adapthist(modified_img_array, clip_limit=0.3)

    # Map threshold constraints to derive solid binary segmentation bounds
    modified_img_array[modified_img_array < 0.7] = 0.0
    modified_img_array[(modified_img_array >= 0.7) & (modified_img_array < 1.0)] = 1.0

    # Execute structural 3x3 median filtering to damp salt-and-pepper artifacts
    modified_img_array = median_filter(modified_img_array, size=3)
    processed_forward_norm.append(modified_img_array)

forward_norm_1 = processed_forward_norm

# =====================================================================
# FILE ARTIFACT EXPORT MATRIX
# =====================================================================
np.save('backwards_small.npy', np.stack(backward_norm))
np.save('forwards_small.npy', np.stack(forward_norm_1))

print(f"\n=====================================================================")
print(f"PIPELINE VERIFICATION STATISTICS:")
print(f"=====================================================================")
print(f"  Backward Array (Input Bounds) -> Min: {np.min(backward_norm):.4f} | Max: {np.max(backward_norm):.4f}")
print(f"  Forward Array (Target Mask)   -> Min: {np.min(forward_norm_1):.4f} | Max: {np.max(forward_norm_1):.4f}")
